# Command extraction debug

In [7]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.style.use('ggplot')

PKG_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PKG_ROOT))
node_fdm_src = PKG_ROOT.parent / 'node-fdm' / 'src'
if node_fdm_src.exists():
    sys.path.insert(0, str(node_fdm_src))

from node_fdm.data.meteo_and_parameters import detect_constant_segments, build_spd_and_vert_selected_from_segments
from pipeline.config import load_config, config_for_node_fdm
from pipeline.frame import merge_adsb_modes, to_node_fdm_frame
from pipeline.opendata import route_dataset_dir

ROUTE_NAME = 'EHAM_LSZH'
ROUTE_DIR = route_dataset_dir(ROUTE_NAME)
ADSB_DIR = ROUTE_DIR / 'data' / 'adsb'
MODES_DIR = ROUTE_DIR / 'data' / 'modes_decoded'

manifest = pd.read_parquet(ROUTE_DIR / 'manifest.parquet')
FLIGHT_IDS = manifest.loc[manifest['status'] == 'done', 'flight_id'].astype(str).tolist()
print(f'{len(FLIGHT_IDS)} flights | route {ROUTE_DIR.name}')

100 flights | route EHAM_LSZH


## CONFIG

In [8]:
CONFIG = load_config()
NODE_CFG = config_for_node_fdm(CONFIG)
display(pd.DataFrame(CONFIG).T)

,tol,min_len,alt_threshold,use_alt,smooth_window,smooth_method,min_abs_value,fill,vz_tol,min_stable_s,quantize,min_alt_ft,bridge_s
mach,0.01,120,20000,True,200,binned,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cas,2.5,60,NaN,False,150,binned,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vz,25,15,NaN,False,50,binned,-1,"{'enabled': True, 'bridge_gaps': True, 'max_ga...",NaN,NaN,NaN,NaN,NaN
h_sel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,250.0,10.0,100.0,NaN,NaN
mach_regime,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28000.0,600.0
add_alt,False,False,False,False,False,False,False,False,False,False,False,False,False


## 1) Load observed (1 Hz grid)

In [9]:
def load_observed(flight_id: str) -> pd.DataFrame:
    adsb = pd.read_parquet(ADSB_DIR / f'{flight_id}.parquet')
    modes = pd.read_parquet(MODES_DIR / f'{flight_id}.parquet')
    out = to_node_fdm_frame(merge_adsb_modes(adsb, modes))
    out['t_s'] = (out['timestamp'] - out['timestamp'].iloc[0]).dt.total_seconds()
    return out

## 2) Plateau detector

In [10]:
def infer_commands(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    return build_spd_and_vert_selected_from_segments(df, config_for_node_fdm(cfg))

## 3) Browse flights

In [11]:
FLIGHT_INDEX = globals().get('FLIGHT_INDEX', 0)
flight_id = FLIGHT_IDS[FLIGHT_INDEX]

In [13]:
def plot_flight(d, title):
    t = d['t_s']
    fig = make_subplots(
        rows=4,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=('Altitude', 'Vertical rate', 'CAS', 'Mach'),
        specs=[[{}], [{"secondary_y": True}], [{"secondary_y": True}], [{"secondary_y": True}]],
    )

    fig.add_trace(go.Scatter(x=t, y=d['altitude'], mode='lines', line=dict(color='gray', width=1), name='observed altitude'), row=1, col=1)
    fig.add_trace(go.Scatter(x=t, y=d['h_sel'], mode='lines', line=dict(color='red', width=1.2, shape='hv'), name='h_sel'), row=1, col=1)

    fig.add_trace(go.Scatter(x=t, y=d['vertical_rate'], mode='lines', line=dict(color='gray', width=1), name='observed VS'), row=2, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=t, y=detect_constant_segments(d, 'vertical_rate', **NODE_CFG['vz'])[1], mode='lines', line=dict(color='orange', width=1.4), name=f"{NODE_CFG['vz']['smooth_method']} ({NODE_CFG['vz']['smooth_window']}s)"), row=2, col=1, secondary_y=False)
    m = d['vz_sel'].notna()
    fig.add_trace(go.Scatter(x=t[m], y=d.loc[m, 'vz_sel'], mode='markers', marker=dict(color='red', size=5), name='vz_sel'), row=2, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(x=t, y=d['CAS'], mode='lines', line=dict(color='gray', width=1), name='observed CAS'), row=3, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=t, y=detect_constant_segments(d, 'CAS', **NODE_CFG['cas'])[1], mode='lines', line=dict(color='orange', width=1.4), name=f"{NODE_CFG['cas']['smooth_method']} ({NODE_CFG['cas']['smooth_window']}s)"), row=3, col=1, secondary_y=False)
    m = d['cas_sel'].notna()
    fig.add_trace(go.Scatter(x=t[m], y=d.loc[m, 'cas_sel'], mode='markers', marker=dict(color='red', size=5), name='cas_sel'), row=3, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(x=t, y=d['Mach'], mode='lines', line=dict(color='gray', width=1), name='observed Mach'), row=4, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=t, y=detect_constant_segments(d, 'Mach', **NODE_CFG['mach'])[1], mode='lines', line=dict(color='orange', width=1.4), name=f"{NODE_CFG['mach']['smooth_method']} ({NODE_CFG['mach']['smooth_window']}s)"), row=4, col=1, secondary_y=False)
    m = d['mach_sel'].notna()
    fig.add_trace(go.Scatter(x=t[m], y=d.loc[m, 'mach_sel'], mode='markers', marker=dict(color='red', size=5), name='mach_sel'), row=4, col=1, secondary_y=False)

    fig.update_layout(height=1100, title_text=title, showlegend=True, hovermode='x unified')
    fig.update_yaxes(title_text='ft', row=1, col=1)
    fig.update_yaxes(title_text='fpm', row=2, col=1, secondary_y=False)
    fig.update_yaxes(title_text='', showticklabels=False, visible=False, row=2, col=1, secondary_y=True)
    fig.update_yaxes(title_text='kt', row=3, col=1, secondary_y=False)
    fig.update_yaxes(title_text='', showticklabels=False, visible=False, row=3, col=1, secondary_y=True)
    fig.update_yaxes(title_text='Mach', row=4, col=1, secondary_y=False)
    fig.update_yaxes(title_text='', showticklabels=False, visible=False, row=4, col=1, secondary_y=True)
    fig.update_xaxes(title_text='time [s]', row=4, col=1)
    display(fig)


output = widgets.Output()
prev_button = widgets.Button(description='← Prev')
next_button = widgets.Button(description='Next →')
index_label = widgets.HTML()


def render_current():
    with output:
        clear_output(wait=True)
        global flight_id, df
        flight_id = FLIGHT_IDS[FLIGHT_INDEX]
        obs = load_observed(flight_id)
        df = infer_commands(obs, CONFIG)
        index_label.value = f'<b>{FLIGHT_INDEX + 1}</b> / {len(FLIGHT_IDS)}: {flight_id}'
        print('flight:', flight_id)
        print('seconds:', int(df['t_s'].max()), '| rows:', len(df))
        for c in ['mach_sel', 'cas_sel', 'vz_sel', 'h_sel']:
            n = df[c].notna().sum()
            print(f'  {c}: {n} samples ({100 * n / len(df):.1f}%)')
        plot_flight(df, flight_id)


def go_prev(_):
    global FLIGHT_INDEX
    FLIGHT_INDEX = (FLIGHT_INDEX - 1) % len(FLIGHT_IDS)
    render_current()


def go_next(_):
    global FLIGHT_INDEX
    FLIGHT_INDEX = (FLIGHT_INDEX + 1) % len(FLIGHT_IDS)
    render_current()


prev_button.on_click(go_prev)
next_button.on_click(go_next)
display(widgets.HBox([prev_button, next_button, index_label]))
display(output)
render_current()

Output()